# 🏆 My Kaggle Solution
**Author:** your-kaggle-username  |  **Task:** binary  |  **Metric:** auc

> *Notebook được generate tự động từ modular pipeline. Thêm nhận xét cá nhân vào các ô markdown.*

---


## ⚙️ 1. Configuration
> Tất cả tham số pipeline — chỉnh tại đây.

In [ ]:
import yaml, json

# Load config (hoặc paste trực tiếp nếu chạy standalone trên Kaggle)
CFG = {'competition': {'name': 'competition-name', 'task': 'binary', 'target_col': 'target', 'id_col': 'id'}, 'paths': {'train': 'data/raw/train.csv', 'test': 'data/raw/test.csv', 'sub': 'data/raw/sample_submission.csv', 'output': 'outputs/'}, 'cv': {'n_folds': 5, 'seed': 42, 'strategy': 'auto', 'group_col': None}, 'metric': 'auc', 'models': {'lgbm': True, 'xgb': True, 'catboost': True}, 'tuning': {'enabled': False, 'n_trials': 50}, 'ensemble': {'blend_n_trials': 300}, 'notebook': {'title': 'My Kaggle Solution', 'author': 'your-kaggle-username', 'output_path': 'notebooks/solution.ipynb'}}

TASK       = CFG["competition"]["task"]
TARGET_COL = CFG["competition"]["target_col"]
ID_COL     = CFG["competition"]["id_col"]
METRIC     = CFG["metric"]
N_FOLDS    = CFG["cv"]["n_folds"]
SEED       = CFG["cv"]["seed"]

print(f"Task   : {TASK}")
print(f"Target : {TARGET_COL}")
print(f"Metric : {METRIC}")
print(f"Folds  : {N_FOLDS}")


## 📦 2. Imports


In [ ]:
import warnings, json, time, gc
from pathlib import Path
from datetime import datetime
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

from sklearn.model_selection import StratifiedKFold, KFold, GroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, log_loss, f1_score, accuracy_score,
    mean_squared_error, mean_absolute_error, r2_score,
)
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRegressor, Pool
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("✅ Imports OK")


## 🛠️ 3. Utility Functions
> Metric dispatcher, CV runner, và helpers.

In [ ]:
import numpy as np
import pandas as pd
import time
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRegressor, Pool
from sklearn.metrics import (
    roc_auc_score, log_loss, f1_score, accuracy_score,
    mean_squared_error, mean_absolute_error, r2_score,
)


# ── Metric ────────────────────────────────────────────────────────────────────
def compute_metric(y_true, y_pred, metric: str, task: str) -> float:
    """Tất cả metrics đều higher = better (loss metrics dùng negative)."""
    if metric == "auc":
        if task == "multiclass":
            return roc_auc_score(y_true, y_pred, multi_class="ovr", average="macro")
        return roc_auc_score(y_true, y_pred)
    elif metric == "logloss":
        return -log_loss(y_true, y_pred)
    elif metric == "rmse":
        return -np.sqrt(mean_squared_error(y_true, y_pred))
    elif metric == "mae":
        return -mean_absolute_error(y_true, y_pred)
    elif metric == "rmsle":
        return -np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))
    elif metric == "r2":
        return r2_score(y_true, y_pred)
    elif metric in ("f1", "f1_macro"):
        avg = "macro" if metric == "f1_macro" else "binary"
        pred_label = (np.argmax(y_pred, axis=1) if np.array(y_pred).ndim > 1
                      else (np.array(y_pred) > 0.5).astype(int))
        return f1_score(y_true, pred_label, average=avg)
    elif metric == "accuracy":
        pred_label = (np.argmax(y_pred, axis=1) if np.array(y_pred).ndim > 1
                      else (np.array(y_pred) > 0.5).astype(int))
        return accuracy_score(y_true, pred_label)
    raise ValueError(f"Unknown metric: {metric}")


# ── Core CV runner ────────────────────────────────────────────────────────────
def run_cv(
    model_fn,
    train: pd.DataFrame,
    target_col: str,
    feature_cols: list[str],
    metric: str,
    task: str,
    label: str = "model",
) -> dict:
    """
    K-fold cross-validation. Trả về OOF predictions + per-fold models.
    model_fn: callable(X_tr, y_tr, X_val, y_val) → fitted model
    """
    fold_ids = sorted(train["fold"].unique())
    n_classes = int(train[target_col].nunique()) if task != "regression" else 1

    if task == "multiclass" and n_classes > 2:
        oof_preds = np.zeros((len(train), n_classes))
    else:
        oof_preds = np.zeros(len(train))

    scores, models = [], []
    t0 = time.time()

    print(f"\n{'─'*55}\n  Training: {label}\n{'─'*55}")

    for fold in fold_ids:
        tr_idx  = train["fold"] != fold
        val_idx = train["fold"] == fold
        X_tr  = train.loc[tr_idx,  feature_cols]
        y_tr  = train.loc[tr_idx,  target_col]
        X_val = train.loc[val_idx, feature_cols]
        y_val = train.loc[val_idx, target_col]

        model = model_fn(X_tr, y_tr, X_val, y_val)

        # Predict
        if task == "regression" or not hasattr(model, "predict_proba"):
            preds = model.predict(X_val)
        else:
            p = model.predict_proba(X_val)
            preds = p[:, 1] if task == "binary" else p

        oof_preds[val_idx.values] = preds
        score = compute_metric(y_val, preds, metric, task)
        scores.append(score)
        models.append(model)
        print(f"  Fold {fold+1}/{len(fold_ids)}  {metric}: {score:.5f}")

    mean_score = float(np.mean(scores))
    std_score  = float(np.std(scores))
    elapsed    = time.time() - t0

    print(f"\n  {'='*50}")
    print(f"  {label:15s} {metric.upper()}: {mean_score:.5f} ± {std_score:.5f}  ⏱ {elapsed:.1f}s")
    print(f"  {'='*50}\n")

    return {
        "oof":    oof_preds,
        "scores": scores,
        "mean":   mean_score,
        "std":    std_score,
        "models": models,
        "label":  label,
    }


# ── LightGBM ──────────────────────────────────────────────────────────────────
def build_lgbm_fn(params: dict, task: str):
    def lgbm_fn(X_tr, y_tr, X_val, y_val):
        Model = lgb.LGBMClassifier if task != "regression" else lgb.LGBMRegressor
        m = Model(**params)
        m.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
        )
        return m
    return lgbm_fn


def get_lgbm_params(task: str, n_classes: int = 2, seed: int = 42) -> dict:
    base = {
        "n_estimators": 1000, "learning_rate": 0.05,
        "num_leaves": 63, "max_depth": -1,
        "min_child_samples": 20,
        "subsample": 0.8, "colsample_bytree": 0.8,
        "reg_alpha": 0.1, "reg_lambda": 1.0,
        "random_state": seed, "n_jobs": -1, "verbose": -1,
    }
    if task == "binary":
        base.update({"objective": "binary", "metric": "auc"})
    elif task == "multiclass":
        base.update({"objective": "multiclass", "metric": "multi_logloss",
                     "num_class": n_classes})
    else:
        base.update({"objective": "regression", "metric": "rmse"})
    return base


# ── XGBoost ───────────────────────────────────────────────────────────────────
def build_xgb_fn(params: dict, task: str):
    def xgb_fn(X_tr, y_tr, X_val, y_val):
        Model = xgb.XGBClassifier if task != "regression" else xgb.XGBRegressor
        m = Model(**params)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        return m
    return xgb_fn


def get_xgb_params(task: str, n_classes: int = 2, seed: int = 42) -> dict:
    base = {
        "n_estimators": 1000, "learning_rate": 0.05,
        "max_depth": 6, "min_child_weight": 5,
        "subsample": 0.8, "colsample_bytree": 0.8,
        "gamma": 0.1, "reg_alpha": 0.1, "reg_lambda": 1.0,
        "random_state": seed, "n_jobs": -1, "verbosity": 0, "tree_method": "hist",
    }
    if task == "binary":
        base.update({"objective": "binary:logistic", "eval_metric": "auc"})
    elif task == "multiclass":
        base.update({"objective": "multi:softprob", "num_class": n_classes})
    else:
        base.update({"objective": "reg:squarederror"})
    return base


# ── CatBoost ──────────────────────────────────────────────────────────────────
def build_cat_fn(params: dict, task: str, cat_feature_indices: list[int]):
    def cat_fn(X_tr, y_tr, X_val, y_val):
        Model = CatBoostClassifier if task != "regression" else CatBoostRegressor
        m = Model(**params)
        m.fit(
            Pool(X_tr, y_tr,  cat_features=cat_feature_indices),
            eval_set=Pool(X_val, y_val, cat_features=cat_feature_indices),
        )
        return m
    return cat_fn


def get_cat_params(task: str, seed: int = 42) -> dict:
    return {
        "iterations": 1000, "learning_rate": 0.05, "depth": 6,
        "l2_leaf_reg": 3, "random_seed": seed, "verbose": False,
        "early_stopping_rounds": 50,
        "eval_metric": ("AUC" if task == "binary"
                        else "MultiClass" if task == "multiclass" else "RMSE"),
    }


# ── Predict test ──────────────────────────────────────────────────────────────
def predict_test(results: dict, test: pd.DataFrame, feature_cols: list[str], task: str) -> np.ndarray:
    """Average predictions từ tất cả fold models."""
    preds_list = []
    for model in results["models"]:
        if task == "regression" or not hasattr(model, "predict_proba"):
            p = model.predict(test[feature_cols])
        else:
            pa = model.predict_proba(test[feature_cols])
            p  = pa[:, 1] if task == "binary" else pa
        preds_list.append(p)
    return np.mean(preds_list, axis=0)

## 📂 4. Load Data


In [ ]:
TRAIN_PATH = CFG["paths"]["train"]
TEST_PATH  = CFG["paths"]["test"]
SUB_PATH   = CFG["paths"]["sub"]

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sub   = pd.read_csv(SUB_PATH)

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")
train.head(3)


## 🔍 5. Exploratory Data Analysis
> Khám phá cấu trúc dữ liệu, missing values, và phân phối target.


In [ ]:
# ── Missing values ────────────────────────────────────────────────────────
missing = train.isnull().mean().mul(100).round(2)
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing):
    print("⚠️  Missing values (%):")
    print(missing.to_string())
else:
    print("✅ No missing values")

print(f"\nDuplicate rows: {train.duplicated().sum()}")
print(f"Shape: {train.shape}")
train.describe()


In [ ]:
# ── Target distribution ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if TASK in ("binary", "multiclass"):
    vc = train[TARGET_COL].value_counts()
    axes[0].bar(vc.index.astype(str), vc.values, color=sns.color_palette("muted"))
    axes[0].set_title("Target Count")
    for i, v in enumerate(vc.values):
        axes[0].text(i, v, f"{v/len(train)*100:.1f}%", ha="center", va="bottom")
    axes[1].pie(vc.values, labels=vc.index.astype(str), autopct="%1.1f%%",
                colors=sns.color_palette("muted"))
    axes[1].set_title("Target Proportion")
else:
    axes[0].hist(train[TARGET_COL], bins=50, color="#5b8dee", edgecolor="white")
    axes[0].set_title("Target Distribution")
    axes[1].boxplot(train[TARGET_COL].dropna())
    axes[1].set_title("Target Boxplot")

plt.suptitle(f"Target: '{TARGET_COL}'", fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# ── Numeric feature distributions ─────────────────────────────────────────
num_cols_eda = train.select_dtypes(include="number").columns.tolist()
num_cols_eda = [c for c in num_cols_eda if c != TARGET_COL][:16]

if num_cols_eda:
    ncols = 4
    nrows = (len(num_cols_eda) + 3) // 4
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows*3))
    axes = axes.flatten()
    for i, col in enumerate(num_cols_eda):
        axes[i].hist(train[col].dropna(), bins=40, color="#5b8dee",
                     edgecolor="white", alpha=0.8)
        axes[i].set_title(col, fontsize=9)
        axes[i].tick_params(labelsize=7)
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle("Numeric Feature Distributions", fontsize=13)
    plt.tight_layout(); plt.show()


In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────
num_corr = train.select_dtypes(include="number").columns.tolist()[:21]
corr = train[num_corr].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, mask=mask, annot=len(num_corr)<=15, fmt=".2f",
            cmap="coolwarm", center=0, square=True,
            linewidths=0.5, ax=ax, annot_kws={"size": 7})
ax.set_title("Correlation Matrix", fontsize=13)
plt.tight_layout(); plt.show()


## 🧹 6. Cleaning & CV Folds
> Drop cột rác + tạo fold trước FE để tránh data leak.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, KFold, GroupKFold


def basic_clean(
    train: pd.DataFrame,
    test: pd.DataFrame,
    target_col: str,
    id_col: str | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Drop constant columns và columns missing > 90%.
    Giữ nguyên target và id.
    """
    protected = {c for c in [target_col, id_col] if c}
    drop_cols = []

    for col in train.columns:
        if col in protected:
            continue
        if train[col].nunique() <= 1:
            drop_cols.append(col)
            print(f"  [Clean] Drop constant col   : {col}")
        elif train[col].isnull().mean() > 0.90:
            pct = train[col].isnull().mean() * 100
            drop_cols.append(col)
            print(f"  [Clean] Drop high-missing col: {col} ({pct:.0f}%)")

    drop_cols = list(set(drop_cols))
    train = train.drop(columns=drop_cols)
    test  = test.drop(columns=[c for c in drop_cols if c in test.columns])
    print(f"[Clean] After cleaning — Train: {train.shape}  Test: {test.shape}")
    return train, test


def make_folds(
    df: pd.DataFrame,
    target_col: str,
    n_folds: int = 5,
    strategy: str = "auto",
    task: str = "binary",
    group_col: str | None = None,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Tạo cột 'fold' cho cross-validation.
    strategy: 'auto' | 'stratified' | 'kfold' | 'group'
    """
    df = df.copy()

    if strategy == "auto":
        strategy = "stratified" if task in ("binary", "multiclass") else "kfold"

    if strategy == "stratified":
        kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
        split_iter = kf.split(df, df[target_col])
    elif strategy == "group" and group_col:
        kf = GroupKFold(n_splits=n_folds)
        split_iter = kf.split(df, df[target_col], df[group_col])
    else:
        kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
        split_iter = kf.split(df)

    df["fold"] = -1
    for fold, (_, val_idx) in enumerate(split_iter):
        df.loc[val_idx, "fold"] = fold

    print(f"[Folds] Strategy: {strategy}  |  {n_folds} folds created")
    counts = df["fold"].value_counts().sort_index()
    for fold, cnt in counts.items():
        print(f"  Fold {fold}: {cnt:,} samples")
    return df

In [ ]:
train, test = basic_clean(train, test, TARGET_COL, ID_COL)
train       = make_folds(train, TARGET_COL, N_FOLDS,
                         strategy=CFG["cv"]["strategy"],
                         task=TASK, seed=SEED)


## 🔧 7. Feature Engineering
> ✏️ **Phần quan trọng nhất — tùy chỉnh theo từng competition.**  
> Các bước imputation + encoding cơ bản đã có sẵn.  
> Thêm domain knowledge vào section `custom_fe()`.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

# ── Track features mới tạo ra ────────────────────────────────────────────────
NEW_FEATURES: list[str] = []


# ═════════════════════════════════════════════════════════════════════════════
# BƯỚC 1 — IMPUTATION (chạy luôn, không cần chỉnh)
# =============================================================================
def impute(
    train: pd.DataFrame,
    test: pd.DataFrame,
    num_cols: list[str],
    cat_cols: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Median impute numeric, most_frequent impute categorical."""
    if num_cols:
        imp = SimpleImputer(strategy="median")
        train[num_cols] = imp.fit_transform(train[num_cols])
        test[num_cols]  = imp.transform(test[num_cols])

    if cat_cols:
        imp = SimpleImputer(strategy="most_frequent")
        train[cat_cols] = imp.fit_transform(train[cat_cols])
        test[cat_cols]  = imp.transform(test[cat_cols])

    print(f"[FE] Imputation done  (num={len(num_cols)}, cat={len(cat_cols)})")
    return train, test


# ═════════════════════════════════════════════════════════════════════════════
# BƯỚC 2 — ENCODING (chạy luôn, không cần chỉnh)
# =============================================================================
def label_encode(
    train: pd.DataFrame,
    test: pd.DataFrame,
    cat_cols: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """Label encode categorical columns. CatBoost sẽ dùng indices riêng."""
    encoders = {}
    for col in cat_cols:
        le = LabelEncoder()
        combined = pd.concat([train[col], test[col]], axis=0).astype(str)
        le.fit(combined)
        train[col] = le.transform(train[col].astype(str))
        test[col]  = le.transform(test[col].astype(str))
        encoders[col] = le
    print(f"[FE] Label encoded {len(cat_cols)} categorical cols")
    return train, test, encoders


# ═════════════════════════════════════════════════════════════════════════════
# BƯỚC 3 — CUSTOM FEATURE ENGINEERING  ← CHỈNH Ở ĐÂY
# =============================================================================
def custom_fe(
    train: pd.DataFrame,
    test: pd.DataFrame,
    target_col: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    ✏️  Thêm custom features vào đây.
    Nhớ append tên feature mới vào NEW_FEATURES.
    """
    global NEW_FEATURES
    NEW_FEATURES = []  # reset mỗi lần chạy

    # ── Interaction features ─────────────────────────────────────────────────
    # Ví dụ: nhân/chia hai cột có ý nghĩa domain
    #
    # for df in [train, test]:
    #     df["feat_A_x_B"] = df["col_A"] * df["col_B"]
    #     df["feat_A_div_B"] = df["col_A"] / (df["col_B"] + 1e-9)
    # NEW_FEATURES += ["feat_A_x_B", "feat_A_div_B"]

    # ── Log transform (skewed features) ─────────────────────────────────────
    #
    # for col in ["col_skewed"]:
    #     for df in [train, test]:
    #         df[f"log_{col}"] = np.log1p(df[col].clip(lower=0))
    #     NEW_FEATURES.append(f"log_{col}")

    # ── GroupBy aggregation ──────────────────────────────────────────────────
    #
    # GROUP_COL = "cat_col"
    # VALUE_COL = "num_col"
    # for agg_fn in ["mean", "std", "max", "min"]:
    #     feat = f"{GROUP_COL}_{VALUE_COL}_{agg_fn}"
    #     stats = train.groupby(GROUP_COL)[VALUE_COL].agg(agg_fn)
    #     train[feat] = train[GROUP_COL].map(stats)
    #     test[feat]  = test[GROUP_COL].map(stats)
    #     NEW_FEATURES.append(feat)

    # ── Target encoding (CV-safe, tránh leak) ───────────────────────────────
    #
    # def target_encode_cv(df, col, target, fold_col, smooth=20):
    #     global_mean = df[target].mean()
    #     out = df[col].copy().astype(float)
    #     for fold in df[fold_col].unique():
    #         tr_mask  = df[fold_col] != fold
    #         val_mask = df[fold_col] == fold
    #         stats = df[tr_mask].groupby(col)[target].agg(["mean","count"])
    #         stats["enc"] = (stats["mean"] * stats["count"] + global_mean * smooth) \
    #                        / (stats["count"] + smooth)
    #         out[val_mask] = df.loc[val_mask, col].map(stats["enc"]).fillna(global_mean)
    #     return out
    #
    # for col in cat_cols:
    #     train[f"te_{col}"] = target_encode_cv(train, col, target_col, "fold")
    #     test[f"te_{col}"]  = test[col].map(
    #         train.groupby(col)[target_col].mean()).fillna(train[target_col].mean())
    #     NEW_FEATURES.append(f"te_{col}")

    # ── Datetime decomposition ───────────────────────────────────────────────
    #
    # if "date_col" in train.columns:
    #     for df in [train, test]:
    #         df["date_col"] = pd.to_datetime(df["date_col"])
    #         df["year"]       = df["date_col"].dt.year
    #         df["month"]      = df["date_col"].dt.month
    #         df["dow"]        = df["date_col"].dt.dayofweek
    #         df["is_weekend"] = (df["dow"] >= 5).astype(int)
    #     NEW_FEATURES += ["year", "month", "dow", "is_weekend"]

    # ── Row-wise stats (nhóm cột cùng loại) ──────────────────────────────────
    #
    # sensor_cols = [c for c in train.columns if c.startswith("sensor_")]
    # if sensor_cols:
    #     for df in [train, test]:
    #         df["sensor_mean"] = df[sensor_cols].mean(axis=1)
    #         df["sensor_std"]  = df[sensor_cols].std(axis=1)
    #         df["sensor_max"]  = df[sensor_cols].max(axis=1)
    #     NEW_FEATURES += ["sensor_mean", "sensor_std", "sensor_max"]

    print(f"[FE] Custom FE done — {len(NEW_FEATURES)} new features: {NEW_FEATURES}")
    return train, test


# ═════════════════════════════════════════════════════════════════════════════
# ENTRY POINT — gọi từ main pipeline
# =============================================================================
def run_feature_engineering(
    train: pd.DataFrame,
    test: pd.DataFrame,
    exclude_cols: list[str],
    target_col: str,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], list[str], list[str]]:
    """
    Chạy toàn bộ FE pipeline.
    Returns: (train, test, feature_cols, cat_cols, new_features)
    """
    num_cols = [c for c in train.select_dtypes(include="number").columns
                if c not in exclude_cols]
    cat_cols = [c for c in train.select_dtypes(include="object").columns
                if c not in exclude_cols]

    train, test          = impute(train, test, num_cols, cat_cols)
    train, test, _       = label_encode(train, test, cat_cols)
    train, test          = custom_fe(train, test, target_col)

    feature_cols = [c for c in train.columns if c not in exclude_cols]
    print(f"[FE] Total features: {len(feature_cols)}")
    return train, test, feature_cols, cat_cols, NEW_FEATURES

In [ ]:
EXCLUDE_COLS = [TARGET_COL, "fold"] + ([ID_COL] if ID_COL else [])

train, test, FEATURE_COLS, CAT_COLS, NEW_FEATURES = run_feature_engineering(
    train, test, EXCLUDE_COLS, TARGET_COL
)

print(f"\nFeature summary:")
print(f"  Total : {len(FEATURE_COLS)}")
print(f"  New FE: {len(NEW_FEATURES)} → {NEW_FEATURES}")


## 🤖 8. Model Training
> Cross-validation với LightGBM · XGBoost · CatBoost.

In [ ]:
results = {}
n_classes = int(train[TARGET_COL].nunique()) if TASK != "regression" else 2


In [ ]:
# ── LightGBM ──────────────────────────────────────────────────────────────
if CFG["models"].get("lgbm", True):
    lgbm_params = get_lgbm_params(TASK, n_classes, SEED)
    lgbm_fn     = build_lgbm_fn(lgbm_params, TASK)
    results["lgbm"] = run_cv(lgbm_fn, train, TARGET_COL,
                              FEATURE_COLS, METRIC, TASK, label="LightGBM")
    results["lgbm"]["test_preds"] = predict_test(results["lgbm"], test, FEATURE_COLS, TASK)


In [ ]:
# ── XGBoost ───────────────────────────────────────────────────────────────
if CFG["models"].get("xgb", True):
    xgb_params = get_xgb_params(TASK, n_classes, SEED)
    xgb_fn     = build_xgb_fn(xgb_params, TASK)
    results["xgb"] = run_cv(xgb_fn, train, TARGET_COL,
                             FEATURE_COLS, METRIC, TASK, label="XGBoost")
    results["xgb"]["test_preds"] = predict_test(results["xgb"], test, FEATURE_COLS, TASK)


In [ ]:
# ── CatBoost ──────────────────────────────────────────────────────────────
if CFG["models"].get("catboost", True):
    cat_indices = [FEATURE_COLS.index(c) for c in CAT_COLS if c in FEATURE_COLS]
    cat_params  = get_cat_params(TASK, SEED)
    cat_fn      = build_cat_fn(cat_params, TASK, cat_indices)
    results["catboost"] = run_cv(cat_fn, train, TARGET_COL,
                                  FEATURE_COLS, METRIC, TASK, label="CatBoost")
    results["catboost"]["test_preds"] = predict_test(results["catboost"], test, FEATURE_COLS, TASK)


## 📊 9. CV Summary & Feature Importance

In [ ]:
# ── CV Summary ────────────────────────────────────────────────────────────
rows = []
for name, res in results.items():
    row = {"model": name,
           f"cv_{METRIC}_mean": round(res["mean"], 5),
           f"cv_{METRIC}_std":  round(res["std"],  5)}
    row.update({f"fold_{i}": round(s, 5) for i, s in enumerate(res["scores"])})
    rows.append(row)
cv_df = pd.DataFrame(rows).set_index("model")
cv_df


In [ ]:
# ── Feature importance (LightGBM) ─────────────────────────────────────────
if "lgbm" in results:
    imps = np.mean([m.feature_importances_ for m in results["lgbm"]["models"]], axis=0)
    fi = pd.DataFrame({"feature": FEATURE_COLS, "importance": imps})
    fi = fi.sort_values("importance", ascending=False).head(30)

    colors = ["#f97316" if f in NEW_FEATURES else "#5b8dee" for f in fi["feature"]]
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(fi["feature"][::-1], fi["importance"][::-1], color=colors[::-1])
    ax.set_title("Feature Importance — top 30  (🟠 = engineered)")
    ax.set_xlabel("Mean importance across folds")
    plt.tight_layout(); plt.show()


## 🎯 10. Hyperparameter Tuning (Optuna)
> Bật `RUN_TUNING = True` khi muốn tìm params tốt hơn.

In [ ]:
RUN_TUNING = CFG["tuning"]["enabled"]
N_TRIALS   = CFG["tuning"]["n_trials"]

if RUN_TUNING:
    def objective(trial):
        p = {
            "n_estimators":      trial.suggest_int("n_estimators", 300, 2000),
            "learning_rate":     trial.suggest_float("lr", 0.01, 0.3, log=True),
            "num_leaves":        trial.suggest_int("num_leaves", 20, 300),
            "max_depth":         trial.suggest_int("max_depth", 3, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
            "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "random_state": SEED, "n_jobs": -1, "verbose": -1,
        }
        p.update({k: v for k, v in lgbm_params.items()
                   if k in ["objective", "metric", "num_class"]})
        res = run_cv(build_lgbm_fn(p, TASK), train, TARGET_COL,
                     FEATURE_COLS, METRIC, TASK, label=f"Trial {trial.number}")
        return res["mean"]

    direction = "minimize" if METRIC in ("logloss","rmse","mae","rmsle") else "maximize"
    study = optuna.create_study(direction=direction,
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    print(f"\n✅ Best {METRIC}: {study.best_value:.5f}")
    print(json.dumps(study.best_params, indent=2))
else:
    print("ℹ️  Tuning skipped. Set tuning.enabled: true in config.")


## 🧩 11. Ensemble
> Optuna tìm blend weights tối ưu.

In [ ]:
import numpy as np
import pandas as pd
import optuna
from src.model import compute_metric

optuna.logging.set_verbosity(optuna.logging.WARNING)


def optimize_blend(
    results: dict,
    y_true: pd.Series,
    metric: str,
    task:   str,
    n_trials: int = 300,
    seed: int = 42,
) -> tuple[dict, float]:
    """
    Tìm blend weights tối ưu giữa OOF predictions.
    Returns: (weights_dict, best_score)
    """
    keys = list(results.keys())
    oofs = [results[k]["oof"] for k in keys]

    if len(keys) == 1:
        print(f"[Validate] Only 1 model — using {keys[0]} directly")
        score = compute_metric(y_true, oofs[0], metric, task)
        return {keys[0]: 1.0}, score

    def objective(trial):
        raw_w = [trial.suggest_float(f"w_{k}", 0.0, 1.0) for k in keys]
        w = np.array(raw_w) / sum(raw_w)
        blended = sum(wi * oi for wi, oi in zip(w, oofs))
        return compute_metric(y_true, blended, metric, task)

    direction = "minimize" if metric in ("logloss", "rmse", "mae", "rmsle") else "maximize"
    study = optuna.create_study(
        direction=direction,
        sampler=optuna.samplers.TPESampler(seed=seed),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    raw_w = np.array([study.best_params[f"w_{k}"] for k in keys])
    best_weights = {k: round(float(w), 4) for k, w in zip(keys, raw_w / raw_w.sum())}
    best_score   = study.best_value

    print(f"\n[Validate] Optimal blend weights:")
    for k, w in best_weights.items():
        solo = compute_metric(y_true, results[k]["oof"], metric, task)
        print(f"  {k:15s}: weight={w:.4f}  solo {metric}={solo:.5f}")
    print(f"  Blend {metric}: {best_score:.5f}")
    return best_weights, best_score


def blend_final(results: dict, weights: dict, task: str) -> np.ndarray:
    """Blend test predictions với weights đã tối ưu."""
    return sum(weights[k] * results[k]["test_preds"] for k in weights)


def cv_summary(results: dict, metric: str) -> pd.DataFrame:
    """In bảng tóm tắt CV scores của tất cả models."""
    rows = []
    for name, res in results.items():
        rows.append({
            "model": name,
            f"cv_{metric}_mean": round(res["mean"], 5),
            f"cv_{metric}_std":  round(res["std"], 5),
            **{f"fold_{i}": round(s, 5) for i, s in enumerate(res["scores"])},
        })
    df = pd.DataFrame(rows).set_index("model")
    print("\n[Validate] CV Summary:")
    print(df.to_string())
    return df

In [ ]:
y_true = train[TARGET_COL]
best_weights, blend_score = optimize_blend(
    results, y_true, METRIC, TASK,
    n_trials=CFG["ensemble"]["blend_n_trials"],
    seed=SEED,
)
final_preds = blend_final(results, best_weights, TASK)


## 📤 12. Submission
> Export file + ghi experiment log.

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime


def make_submission(
    final_preds: np.ndarray,
    sub_template: pd.DataFrame,
    target_col: str,
    task: str,
    output_dir: str = "outputs/",
    filename: str | None = None,
) -> tuple[pd.DataFrame, Path]:
    """Tạo submission CSV đúng format."""
    sub = sub_template.copy()

    if task == "multiclass" and np.array(final_preds).ndim > 1:
        n_cls = final_preds.shape[1]
        for i in range(n_cls):
            sub[f"class_{i}"] = final_preds[:, i]
    else:
        sub[target_col] = final_preds

    ts       = datetime.now().strftime("%Y%m%d_%H%M%S")
    fname    = filename or f"submission_{ts}.csv"
    filepath = Path(output_dir) / fname
    filepath.parent.mkdir(parents=True, exist_ok=True)
    sub.to_csv(filepath, index=False)

    print(f"[Inference] Submission saved → {filepath}")
    print(sub.head(3).to_string())
    return sub, filepath


def log_experiment(
    cfg:          dict,
    results:      dict,
    weights:      dict,
    blend_score:  float,
    feature_cols: list[str],
    new_features: list[str],
    sub_path:     Path,
    lb_score:     float | None = None,
    notes:        str = "",
) -> pd.DataFrame:
    """
    Ghi lại experiment vào outputs/experiment_log.csv.
    Điền lb_score sau khi submit lên Kaggle.
    """
    metric     = cfg["metric"]
    output_dir = Path(cfg["paths"]["output"])

    entry = {
        "timestamp":     datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "task":          cfg["competition"]["task"],
        "metric":        metric,
        "n_folds":       cfg["cv"]["n_folds"],
        "n_features":    len(feature_cols),
        "n_new_features":len(new_features),
        "new_features":  str(new_features),
        "models":        str(list(weights.keys())),
        "weights":       str(weights),
        f"blend_{metric}": round(blend_score, 5),
        "lb_score":      lb_score,   # ← fill sau khi submit
        "sub_file":      str(sub_path.name),
        "notes":         notes,
    }

    # Per-model scores
    for name, res in results.items():
        entry[f"{name}_{metric}"] = round(res["mean"], 5)

    log_path = output_dir / "experiment_log.csv"
    log_df   = pd.read_csv(log_path) if log_path.exists() else pd.DataFrame()
    log_df   = pd.concat([log_df, pd.DataFrame([entry])], ignore_index=True)
    log_df.to_csv(log_path, index=False)

    print(f"\n[Inference] Experiment logged → {log_path}")
    print(json.dumps({k: v for k, v in entry.items()
                       if k not in ("weights", "new_features", "notes")}, indent=2))
    return log_df

In [ ]:
sub_df, sub_path = make_submission(
    final_preds, sub,
    target_col=TARGET_COL,
    task=TASK,
    output_dir=CFG["paths"]["output"],
)

log_df = log_experiment(
    cfg=CFG,
    results=results,
    weights=best_weights,
    blend_score=blend_score,
    feature_cols=FEATURE_COLS,
    new_features=NEW_FEATURES,
    sub_path=sub_path,
    lb_score=None,    # ← điền sau khi submit
    notes="",         # ← ghi chú experiment
)

print("\n🏁 Done! Good luck on the leaderboard 🎯")
